# Printer Agent - Control Center

This notebook is the **agent + admin control center** for the Sharp Printer Automation System.

It does **not** duplicate logic from `app.py`. Instead, it:

- Imports tools from `tools/` (same tools `app.py` uses)
- Reads printer inventory from `printer_list.py`
- Calls FastAPI endpoints (`/api/printers`, `/api/register`, `/api/logs`)
- Optionally runs a LangChain agent that uses these tools

### Make sure FastAPI is running first:

```powershell
python -m uvicorn app:app --host 0.0.0.0 --port 8000 --reload
```

## 1. Setup & Project Path

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath(".")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

## 2. Import Project Tools

In [ ]:
from printer_list import PRINTERS

from tools.printer_tool import (
    get_all_printers,
    get_printer,
    get_printer_by_ip,
)

try:
    from tools.logger_tool import read_install_logs
    LOGGER_AVAILABLE = True
except Exception as e:
    LOGGER_AVAILABLE = False
    print("Logger not available yet:", e)

print("Total printers in inventory:", len(PRINTERS))

## 3. List All Available Printers

In [ ]:
printers = get_all_printers()

print(f"Total printers: {len(printers)}\n")

for p in printers:
    print(f"{p['id']:35} {p['name']:35} {p['location']:25} {p['ip']}")

## 4. Look Up a Specific Printer

In [ ]:
get_printer("19A_PRINTER")

In [ ]:
get_printer_by_ip("172.16.16.31")

## 5. Verify FastAPI Endpoint `/api/printers`

In [ ]:
import requests

BASE_URL = "http://127.0.0.1:8000"

try:
    response = requests.get(f"{BASE_URL}/api/printers", timeout=5)
    print("API status:", response.status_code)

    data = response.json()
    print(f"Printers returned: {len(data)}\n")

    for p in data[:5]:
        print(p)
except Exception as e:
    print("FastAPI not reachable. Is uvicorn running?")
    print("Error:", e)

## 6. Test Registration via FastAPI

This calls `/api/register` exactly the same way `index.html` does.

In [ ]:
payload = {
    "printer": "19A_PRINTER",
    "name": "Hassan Abdulmalik",
    "user_number": "34451",
    "email": "hassan@dangote.com"
}

res = requests.post(f"{BASE_URL}/api/register", json=payload)

print("Status:", res.status_code)
print("Response:")
print(res.json())

## 7. Read Install Logs

In [ ]:
if LOGGER_AVAILABLE:
    logs = read_install_logs(limit=20)
    for row in logs:
        print(row)
else:
    print("tools/logger_tool.py is not available yet.")

In [ ]:
try:
    res = requests.get(f"{BASE_URL}/api/logs")
    print("Status:", res.status_code)
    print("Logs:")
    for row in res.json():
        print(row)
except Exception as e:
    print("Could not fetch logs from API.")
    print("Error:", e)

## 8. (Optional) Simple LangChain Agent Using Real Tools

This agent uses **your real printer inventory** + **real registration API**, not hardcoded mock data.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent

from tools.printer_tool import (
    get_all_printers,
    get_printer,
    get_printer_by_ip,
)

In [ ]:
@tool
def find_printer(query: str) -> str:
    """Find a Sharp printer by ID, IP, or location/name keyword."""

    # 1. Try direct ID
    p = get_printer(query)
    if p:
        return (
            f"Found printer:\n"
            f"- Name: {p.get('display_name', query)}\n"
            f"- IP: {p['ip']}\n"
            f"- Location: {p.get('location', 'N/A')}"
        )

    # 2. Try IP
    p = get_printer_by_ip(query)
    if p:
        return (
            f"Found printer by IP:\n"
            f"- Name: {p['name']}\n"
            f"- IP: {p['ip']}\n"
            f"- Location: {p['location']}"
        )

    # 3. Fuzzy match by name/location
    matches = []
    for printer in get_all_printers():
        if (
            query.lower() in printer["name"].lower()
            or query.lower() in printer["location"].lower()
        ):
            matches.append(printer)

    if not matches:
        available = ", ".join(p["id"] for p in get_all_printers())
        return f"No printer found matching '{query}'. Available IDs: {available}"

    lines = [f"Found {len(matches)} matches:"]
    for m in matches:
        lines.append(f"- {m['id']} | {m['name']} | {m['location']} | {m['ip']}")
    return "\n".join(lines)


@tool
def register_user(printer_id: str, user_name: str, user_code: str, email: str) -> str:
    """Register a user on a Sharp printer via the FastAPI service."""

    import requests

    res = requests.post(
        "http://127.0.0.1:8000/api/register",
        json={
            "printer": printer_id,
            "name": user_name,
            "user_number": user_code,
            "email": email
        },
        timeout=15
    )

    if res.status_code != 200:
        return f"Registration failed: {res.status_code} {res.text}"

    data = res.json()
    return data.get("message", "Registration completed.")


tools = [find_printer, register_user]

print("Agent tools ready.")

In [ ]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2,
)

agent = create_agent(
    model,
    tools,
    system_prompt=(
        "You are the Sharp Printer Installation Agent for Dangote Refinery IT.\n\n"
        "Rules:\n"
        "1. Always use the 'find_printer' tool to locate the correct printer.\n"
        "2. Always confirm the printer name, location, and IP before registering.\n"
        "3. Before calling 'register_user', make sure you have:\n"
        "   - printer_id\n"
        "   - full name\n"
        "   - user code (5-8 digits)\n"
        "   - email\n"
        "4. Be professional and concise."
    )
)

print("Agent ready.")

## 9. Test the Agent (end-to-end)

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "I need a printer at GED OFFICE"}
    ]
})

for msg in result["messages"]:
    print(f"\n{msg.type}: {msg.content}")

In [ ]:
result2 = agent.invoke({
    "messages": result["messages"] + [
        {
            "role": "user",
            "content": "My name is Hassan Abdulmalik, code 34451, email hassan@dangote.com"
        }
    ]
})

for msg in result2["messages"]:
    print(f"\n{msg.type}: {msg.content}")

## ✅ End

What this notebook now does:

- Uses **the same `printer_list.py`** as `app.py`
- Uses **the same tools** as `app.py`
- Calls **the same FastAPI endpoints** as `index.html`
- Provides an **optional AI agent** that uses the real registration flow

Next possible upgrades:
- Build a small admin dashboard inside the notebook (logs viewer)
- Add a `find_user` tool to search install logs
- Plug Hermes in as the central agent orchestrator